# 7.4: Training a toy model, then greedy decoding and beam search


In [ ]:
import math
import torch
import torch.nn as nn

PAD, BOS, EOS, V = 0, 1, 2, 13

class Seq2Seq(nn.Module):
    def __init__(self, d=64, h=4, N=2, max_len=40):
        super().__init__()
        self.emb = nn.Embedding(V, d)
        nn.init.normal_(self.emb.weight, std=d ** -0.5)             # tied + scaled by sqrt(d)
        pos, omega = torch.arange(max_len)[:, None], 10000 ** (-torch.arange(0, d, 2) / d)
        pe = torch.zeros(max_len, d)
        pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * omega), torch.cos(pos * omega)
        self.register_buffer("pe", pe)
        self.tf = nn.Transformer(d, h, N, N, 4 * d, dropout=0.0, batch_first=True)
        self.d = d
    def embed(self, x):
        return self.emb(x) * math.sqrt(self.d) + self.pe[: x.size(1)]
    def encode(self, src):
        return self.tf.encoder(self.embed(src))                      # run once per source
    def decode(self, tgt_in, memory):
        causal = nn.Transformer.generate_square_subsequent_mask(tgt_in.size(1))
        return self.tf.decoder(self.embed(tgt_in), memory, tgt_mask=causal) @ self.emb.weight.T
    def forward(self, src, tgt_in):
        return self.decode(tgt_in, self.encode(src))

def batch(B=64, lo=4, hi=10):
    L = int(torch.randint(lo, hi + 1, ()))
    src = torch.randint(3, V, (B, L))
    tgt = src.flip(1)
    return src, torch.cat([torch.full((B, 1), BOS), tgt], 1), torch.cat([tgt, torch.full((B, 1), EOS)], 1)

torch.manual_seed(0)
model = Seq2Seq()
opt = torch.optim.Adam(model.parameters(), lr=1e-3, betas=(0.9, 0.98), eps=1e-9)
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
for step in range(1500):                                             # about a minute on a CPU
    src, tgt_in, labels = batch()
    loss = loss_fn(model(src, tgt_in).reshape(-1, V), labels.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
model.eval()

@torch.no_grad()
def greedy(model, src, max_len=30):
    memory = model.encode(src)
    ys = [BOS]
    for _ in range(max_len):
        logits = model.decode(torch.tensor([ys]), memory)[0, -1]
        ys.append(int(logits.argmax()))
        if ys[-1] == EOS:
            break
    return ys[1:]

@torch.no_grad()
def beam_search(model, src, k=4, alpha=0.6, max_len=30):
    memory = model.encode(src)
    lp = lambda n: ((5 + n) / 6) ** alpha                            # Wu et al. length penalty
    beams, finished = [([BOS], 0.0)], []
    for _ in range(max_len):
        prefixes = torch.tensor([seq for seq, _ in beams])           # all live beams in one batch
        logp = model.decode(prefixes, memory.expand(len(beams), -1, -1))[:, -1].log_softmax(-1)
        cands = [(seq + [tok], score + float(logp[i, tok]))
                 for i, (seq, score) in enumerate(beams)
                 for tok in logp[i].topk(k).indices.tolist()]
        cands.sort(key=lambda c: c[1], reverse=True)
        beams = []
        for seq, score in cands:
            if seq[-1] == EOS:
                finished.append((seq[1:], score / lp(len(seq) - 1)))
            else:
                beams.append((seq, score))
            if len(beams) == k:
                break
        if len(finished) >= k or not beams:
            break
    finished += [(seq[1:], score / lp(len(seq) - 1)) for seq, score in beams]
    return max(finished, key=lambda f: f[1])[0]

torch.manual_seed(1)
tests = [torch.randint(3, V, (1, int(torch.randint(4, 11, ())))) for _ in range(200)]
target = lambda s: s[0].flip(0).tolist() + [EOS]
acc_g = sum(greedy(model, s) == target(s) for s in tests) / len(tests)
acc_b = sum(beam_search(model, s) == target(s) for s in tests) / len(tests)
print(f"exact match: greedy {acc_g:.2f}, beam(k=4) {acc_b:.2f}")
print(all(beam_search(model, s, k=1) == greedy(model, s) for s in tests[:20]))  # k=1 is greedy
